# I. Raw Data Acquisition and Initial EDA

This notebook acquires (or reuses) the raw marketing campaign dataset and performs first-pass exploratory data analysis on the raw file.
Its output is `../dataset/marketing_campaign_dataset.csv`, which is used by notebook 2 for cleaning and validation.

## Step 1: Imports and Display Settings

In [1]:
## Core imports for filesystem ops, array handling, and tabular analysis.
from pathlib import Path
import shutil

import numpy as np
import pandas as pd

## Wider display settings keep EDA tables readable in notebook output.
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

## Step 2: Acquire Raw Data

If the raw file already exists, this step reuses it. Otherwise, it downloads from Kaggle and copies the CSV into `../dataset`.

In [2]:
## Keep a deterministic raw-data path consumed by downstream notebooks.
## With FORCE_DOWNLOAD=False, reruns reuse local data instead of redownloading.
# Set True only when you explicitly want to refresh from Kaggle.
FORCE_DOWNLOAD = False
DATASET_DIR = Path("../dataset")
RAW_DATASET_PATH = DATASET_DIR / "marketing_campaign_dataset.csv"

## Create the dataset folder on first run or clean clone.
DATASET_DIR.mkdir(parents=True, exist_ok=True)

if RAW_DATASET_PATH.exists() and not FORCE_DOWNLOAD:
    print(f"Raw dataset already available at: {RAW_DATASET_PATH.resolve()}")
else:
    try:
        ## Lazy import avoids requiring kagglehub when local raw data already exists.
        import kagglehub
    except ImportError as exc:
        raise ImportError("kagglehub is required. Install with: pip install kagglehub") from exc

    kaggle_path = Path(
        kagglehub.dataset_download("manishabhatt22/marketing-campaign-performance-dataset")
    )

    source_csv = kaggle_path / "marketing_campaign_dataset.csv"
    if not source_csv.exists():
        ## Fallback in case Kaggle package layout changes.
        csv_candidates = sorted(kaggle_path.rglob("*.csv"))
        if not csv_candidates:
            raise FileNotFoundError(f"No CSV files found in Kaggle download path: {kaggle_path}")
        source_csv = csv_candidates[0]

    ## Copy to a project-local path so downstream notebooks use a stable source.
    shutil.copy2(source_csv, RAW_DATASET_PATH)
    print(f"Raw dataset copied to: {RAW_DATASET_PATH.resolve()}")

Raw dataset already available at: C:\Users\hecto\OneDrive\Documentos\VSCode\py_marketing_eda\dataset\marketing_campaign_dataset.csv


## Step 3: Load and Preview Raw Dataset

In [3]:
## Load the raw CSV and emit quick provenance and shape checks.
df_raw = pd.read_csv(RAW_DATASET_PATH)

print(f"Raw dataframe shape: {df_raw.shape}")
print(f"Source file: {RAW_DATASET_PATH.resolve()}")
## `head()` provides a fast sanity sample before deeper profiling.
df_raw.head()

Raw dataframe shape: (200000, 16)
Source file: C:\Users\hecto\OneDrive\Documentos\VSCode\py_marketing_eda\dataset\marketing_campaign_dataset.csv


,Campaign_ID,Company,Campaign_Type,Target_Audience,Duration,Channel_Used,Conversion_Rate,Acquisition_Cost,ROI,Location,Language,Clicks,Impressions,Engagement_Score,Customer_Segment,Date
0,1,Innovate Industries,Email,Men 18-24,30 days,Google Ads,0.04,"$16,174.00",6.29,Chicago,Spanish,506,1922,6,Health & Wellness,2021-01-01
1,2,NexGen Systems,Email,Women 35-44,60 days,Google Ads,0.12,"$11,566.00",5.61,New York,German,116,7523,7,Fashionistas,2021-01-02
2,3,Alpha Innovations,Influencer,Men 25-34,30 days,YouTube,0.07,"$10,200.00",7.18,Los Angeles,French,584,7698,1,Outdoor Adventurers,2021-01-03
3,4,DataTech Solutions,Display,All Ages,60 days,YouTube,0.11,"$12,724.00",5.55,Miami,Mandarin,217,1820,7,Health & Wellness,2021-01-04
4,5,NexGen Systems,Email,Men 25-34,15 days,YouTube,0.05,"$16,452.00",6.50,Los Angeles,Mandarin,379,4201,3,Health & Wellness,2021-01-05


## Step 4: Schema and Data Quality Checks

This step checks datatypes, missingness, and duplicates before any transformations are applied.

In [4]:
## `info(show_counts=True)` combines dtype audit and non-null coverage.
df_raw.info(show_counts=True)

## Pair missing counts with percentages to compare absolute and relative impact.
missing_count = df_raw.isna().sum()
missing_pct = (df_raw.isna().mean() * 100).round(2)
missing_summary = (
    pd.DataFrame({"missing_count": missing_count, "missing_pct": missing_pct})
    .sort_values(by=["missing_count", "missing_pct"], ascending=False)
)

## Duplicate rows are measured on raw data before any transformations.
duplicate_rows = int(df_raw.duplicated().sum())
print(f"Duplicate rows: {duplicate_rows}")
missing_summary

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Campaign_ID       200000 non-null  int64  
 1   Company           200000 non-null  str    
 2   Campaign_Type     200000 non-null  str    
 3   Target_Audience   200000 non-null  str    
 4   Duration          200000 non-null  str    
 5   Channel_Used      200000 non-null  str    
 6   Conversion_Rate   200000 non-null  float64
 7   Acquisition_Cost  200000 non-null  str    
 8   ROI               200000 non-null  float64
 9   Location          200000 non-null  str    
 10  Language          200000 non-null  str    
 11  Clicks            200000 non-null  int64  
 12  Impressions       200000 non-null  int64  
 13  Engagement_Score  200000 non-null  int64  
 14  Customer_Segment  200000 non-null  str    
 15  Date              200000 non-null  str    
dtypes: float64(2), int64(4), str(10

,missing_count,missing_pct
Campaign_ID,0,0.0
Company,0,0.0
Campaign_Type,0,0.0
Target_Audience,0,0.0
Duration,0,0.0
Channel_Used,0,0.0
Conversion_Rate,0,0.0
Acquisition_Cost,0,0.0
ROI,0,0.0
Location,0,0.0


## Step 5: Numeric and Categorical Profiling

Use this summary to identify high-variance fields, outlier candidates, and categorical imbalance in raw data.

In [5]:
## Split numeric and non-numeric columns so each profile uses fit-for-purpose stats.
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_raw.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_profile = df_raw[numeric_cols].describe().T if numeric_cols else pd.DataFrame()
if not numeric_profile.empty:
    ## Rank by standard deviation to surface high-variance raw features.
    numeric_profile = numeric_profile.sort_values(by="std", ascending=False)

## For categorical fields, keep cardinality plus dominant value/frequency.
categorical_profile = pd.DataFrame(
    {
        "n_unique": df_raw[cat_cols].nunique(dropna=False),
        "top": [df_raw[c].mode(dropna=False).iloc[0] if not df_raw[c].mode(dropna=False).empty else np.nan for c in cat_cols],
        "top_freq": [df_raw[c].value_counts(dropna=False).iloc[0] if not df_raw[c].value_counts(dropna=False).empty else np.nan for c in cat_cols],
    }
) if cat_cols else pd.DataFrame()

print("Numeric profile (top 15 by standard deviation):")
display(numeric_profile.head(15) if not numeric_profile.empty else pd.DataFrame({"message": ["No numeric columns found"]}))

print("Categorical profile:")
display(categorical_profile if not categorical_profile.empty else pd.DataFrame({"message": ["No categorical columns found"]}))

Numeric profile (top 15 by standard deviation):


,count,mean,std,min,25%,50%,75%,max
Campaign_ID,200000.0,100000.500000,57735.171256,1.00,50000.75,100000.50,150000.25,200000.00
Impressions,200000.0,5507.301520,2596.864286,1000.00,3266.00,5517.50,7753.00,10000.00
Clicks,200000.0,549.772030,260.019056,100.00,325.00,550.00,775.00,1000.00
Engagement_Score,200000.0,5.494710,2.872581,1.00,3.00,5.00,8.00,10.00
ROI,200000.0,5.002438,1.734488,2.00,3.50,5.01,6.51,8.00
Conversion_Rate,200000.0,0.080070,0.040602,0.01,0.05,0.08,0.12,0.15


Categorical profile:


,n_unique,top,top_freq
Company,5,TechCorp,40237
Campaign_Type,5,Influencer,40169
Target_Audience,5,Men 18-24,40258
Duration,4,30 days,50255
Channel_Used,6,Email,33599
Acquisition_Cost,15001,"$16,578.00",32
Location,5,Miami,40269
Language,5,Mandarin,40255
Customer_Segment,5,Foodies,40208
Date,365,2021-01-01,548


## Summarized data

In [6]:
## Lightweight text audit for logs/CI where rich notebook display is unavailable.
## Recompute summaries locally so this cell stays self-contained.
# Temporary audit cell for concise notebook-1 insights
missing_summary_local = (
    pd.DataFrame({
        "missing_count": df_raw.isna().sum(),
        "missing_pct": (df_raw.isna().mean() * 100).round(4),
    })
    .sort_values(["missing_count", "missing_pct"], ascending=False)
)

## Use the same std-based ranking criterion as the main profiling cell.
numeric_profile_local = df_raw.select_dtypes(include=[np.number]).describe().T
numeric_profile_local = numeric_profile_local.sort_values("std", ascending=False)

cat_cols_local = df_raw.select_dtypes(exclude=[np.number]).columns.tolist()
cat_profile_local = []
## Capture top-value dominance per categorical column for quick skew checks.
for c in cat_cols_local:
    vc = df_raw[c].value_counts(dropna=False)
    cat_profile_local.append((c, str(vc.index[0]), int(vc.iloc[0]), round(vc.iloc[0] / len(df_raw) * 100, 2)))

print("TOP_MISSING")
print(missing_summary_local.head(5).to_string())
print("\nTOP_NUMERIC_STD")
print(numeric_profile_local[["mean", "std", "min", "max"]].head(5).to_string())
print("\nTOP_CATEGORICAL_DOMINANCE")
print(pd.DataFrame(cat_profile_local, columns=["column", "top_value", "top_freq", "top_pct"]).sort_values("top_pct", ascending=False).head(8).to_string(index=False))

TOP_MISSING
                 missing_count  missing_pct
Campaign_ID                  0          0.0
Company                      0          0.0
Campaign_Type                0          0.0
Target_Audience              0          0.0
Duration                     0          0.0

TOP_NUMERIC_STD
                           mean           std     min       max
Campaign_ID       100000.500000  57735.171256     1.0  200000.0
Impressions         5507.301520   2596.864286  1000.0   10000.0
Clicks               549.772030    260.019056   100.0    1000.0
Engagement_Score       5.494710      2.872581     1.0      10.0
ROI                    5.002438      1.734488     2.0       8.0

TOP_CATEGORICAL_DOMINANCE
          column  top_value  top_freq  top_pct
        Duration    30 days     50255    25.13
 Target_Audience  Men 18-24     40258    20.13
        Language   Mandarin     40255    20.13
        Location      Miami     40269    20.13
         Company   TechCorp     40237    20.12
Customer_Segme

## Notes:

- No missing values in any columns.
- Impressions, Clicks and Engagement_Score show the highest spread. (Campaign_ID not taken into account)
- More analysis on categorical dominance and distribution is needed.